In [ ]:
import geopandas as gpd
import joblib
import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
from matplotlib_scalebar.scalebar import ScaleBar
from scipy import stats
from sklearn import metrics

In [ ]:
census = gpd.read_parquet(
    "/data/uscuni-restricted/04_spatial_census/_merged_census_2021_relative_scaled.parquet"
)

selection = [
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední vč. vyučení bez maturity - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: - střední s maturitou vč. nástavbového a pomaturitního - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání:  vysokoškolské - celkem",
    "Obyvatelstvo - věk: 15 a více - nejvyšší dosažené vzdělání: nezjištěno - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: průmysl - celkem",
    "Zaměstnaní - Pracovníci ve službách a prodeji",
    "Zaměstnaní - Řemeslníci a opraváři",
    "Obyvatelstvo - zaměstnaní - postavení v zaměstnání: zaměstnanci - celkem",
    "Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    "Počet osob v bytech celkem  s právním důvodem užívání: nájemní / pronajatý",
    "Počet osob v bytech celkem  s právním důvodem užívání: družstevní",
    "Počet obyvatel na byt",
    "Počet osob v domech celkem s vlastnictvím:  fyzická osoba",
    "Obyvatelstvo - věk: 0 - 6  - celkem",
    "Obyvatelstvo - věk: 7 - 14  - celkem",
    "Obyvatelstvo - věk: 15 - 24  - celkem",
    "Obyvatelstvo - věk: 45 - 54  - celkem",
    "Obyvatelstvo - ekon. aktivita: osoby na rodičovské dovolené - celkem",
    "Obyvatelstvo - státní občanství: Slovenská republika - celkem",
    "Obyvatelstvo - státní občanství: země EU mimo ČR - celkem",
    "Obyvatelstvo - státní občanství: nezjištěno - celkem",
    "Obyvatelstvo - náboženská víra: bez náboženské víry - celkem",
    "Obyvatelstvo - náboženská víra: neuvedeno - celkem",
    "Obyvatelstvo - s trvalým pobytem - celkem",
    "Obyvatelstvo - rodinný stav: ženatí, vdané - celkem",
    "Obyvatelstvo - rodinný stav: rozvedení - celkem",
    "Obyvatelstvo - rodinný stav: ovdovělí - celkem",
    "geometry",
]
fas = census[selection]
clusters = pd.read_csv(
    "/data/uscuni-restricted/04_spatial_census/cluster_assignment_v10.csv",
    dtype={"kod_nadzsj_d": str},
)
cluster_mapping = pd.read_parquet(
    "/data/uscuni-ulce/processed_data/clusters/cluster_mapping_v10.pq"
)
data = fas.merge(clusters, left_on="nadzsjd", right_on="kod_nadzsj_d")
variables = data.columns.drop(["geometry", "kod_nadzsj_d", "final_without_noise"])

mapped = data["final_without_noise"].map(cluster_mapping[3])

In [ ]:
models_lr = []

for reduction in ["fa"]:
    for model_type in ["lr"]:
        for cluster in [5, 4]:
            path = f"/data/uscuni-restricted/06_models/{reduction}/label_{cluster}/{model_type}/model.joblib"
            with open(path, "rb") as f:
                model = joblib.load(f)
            f1_values = model.local_metric(
                metrics.f1_score, average="macro", zero_division=0
            )

            models_lr.append(
                {
                    "reduction": reduction,
                    "model_type": model_type,
                    "cluster": cluster,
                    "model": model,
                    "f1_values": model.local_metric(
                        metrics.f1_score, average="macro", zero_division=0
                    ),
                }
            )

df_models_lr = pd.DataFrame(models_lr)

In [ ]:
models_rf = []

for reduction in ["fa"]:
    for model_type in ["rf"]:
        for cluster in [5, 4]:
            path = f"/data/uscuni-restricted/06_models/{reduction}/label_{cluster}/{model_type}/model.joblib"
            with open(path, "rb") as f:
                model = joblib.load(f)
            f1_values = model.local_metric(
                metrics.f1_score, average="macro", zero_division=0
            )

            models_rf.append(
                {
                    "reduction": reduction,
                    "model_type": model_type,
                    "cluster": cluster,
                    "model": model,
                    "f1_values": model.local_metric(
                        metrics.f1_score, average="macro", zero_division=0
                    ),
                }
            )

df_models_rf = pd.DataFrame(models_rf)

In [ ]:
coef_cols = [
    "Počet osob v bytech celkem  s právním důvodem užívání: v osobním vlastnictví",
    "Obyvatelstvo - zaměstnaní - odvětví ekon.čin.: zemědělství, lesnictví ,rybářství - celkem",
]

n_clusters = len(df_models_lr)

In [ ]:
custom_coolwarm = LinearSegmentedColormap.from_list(
    "custom_coolwarm", ["#4069BC", "#DDDCDC", "#E0665F"]
)
custom_fi = LinearSegmentedColormap.from_list(
    "custom_fi",
    [
        "#1F943E",
        "#b2cd32",
        "#FEFFBE",
        "#E69C63",
        "#E0665F",
    ],
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))


# loop over clusters and variables
for row_i, (_, row) in enumerate(df_models_lr.iterrows()):
    model = row["model"]
    cluster = row["cluster"]

    for col_i, coef_col in enumerate(coef_cols):
        ax = axes[col_i, row_i]

        series = model.local_coef_[coef_col]
        tmp = data.assign(_coef_tmp=series.values)

        tmp.plot(
            column="_coef_tmp",
            ax=ax,
            cmap=custom_coolwarm,
            vmin=-1,
            vmax=1,
            legend=False,
            missing_kwds={"color": "lightgray"},
        )
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

axes[0][0].set_title("Incoherent Small-Scale Compact Fabric", fontsize=12)
axes[0][0].set_ylabel("Res. in Owner-Occupied Dwellings", fontsize=10)
axes[0][1].set_title("Incoherent Small-Scale Sparse Fabric", fontsize=12)
axes[1][0].set_ylabel("Empl. sector - Agriculture/Forestry/Fishery", fontsize=10)
# axes[1][0].add_artist(ScaleBar(1, location="lower left", height_fraction=0.015))


# Colorbar in col 4, row 2 (last subplot)
cax = fig.add_axes((0.002, 0.2, 0.02, 0.6))
sm = mpl.cm.ScalarMappable(cmap=custom_coolwarm, norm=mpl.colors.Normalize())
sm.set_array([-1, 1])
fig.colorbar(sm, cax=cax)
# plt.tight_layout()
fig.subplots_adjust(wspace=0, hspace=0)
# fig.align_titles()

# plt.tight_layout()
fig.savefig("side_by_side.png", dpi=300, bbox_inches="tight")

In [ ]:
sns.set_context("paper", font_scale=1.5)

In [ ]:
kraje = gpd.read_file("/data/uscuni-kindergartens/geometrie/kraje.geojson").to_crs(
    data.crs
)
stat = kraje.dissolve()

In [ ]:
fig, axes = plt.subplots(n_clusters, 4, figsize=(24, 8))

# GWLR coefficients
for row_i, (_, row) in enumerate(df_models_lr.iterrows()):
    model = row["model"]
    for col_i, coef_col in enumerate(coef_cols):
        ax = axes[col_i, row_i]
        series = model.local_coef_[coef_col]
        tmp = data.assign(_coef_tmp=series.values)
        tmp.plot(
            column="_coef_tmp",
            ax=ax,
            cmap=custom_coolwarm,
            vmin=-1,
            vmax=1,
            legend=False,
            missing_kwds={"color": "lightgray"},
        )
        # Overlay the country boundary
        stat.plot(ax=ax, color="none", edgecolor="black", linewidth=0.3, zorder=3)

        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

# GWRF feature importance
for row_i, (_, row) in enumerate(df_models_rf.iterrows()):
    model = row["model"]
    for col_i, coef_col in enumerate(coef_cols):
        ax = axes[col_i, row_i + 2]
        series = model.feature_importances_[coef_col]
        tmp = data.assign(_coef_tmp=series.values)
        tmp.plot(
            column="_coef_tmp",
            ax=ax,
            cmap=custom_fi,
            vmin=0,
            vmax=0.15,
            legend=False,
            missing_kwds={"color": "lightgray"},
        )
        # Overlay the country boundary
        stat.plot(ax=ax, color="none", edgecolor="black", linewidth=0.3, zorder=3)

        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

axes[0][0].set_title("Incoherent Small-Scale Compact Fabric")
axes[0][1].set_title("Incoherent Small-Scale Sparse Fabric")
axes[0][2].set_title("Incoherent Small-Scale Compact Fabric")
axes[0][3].set_title("Incoherent Small-Scale Sparse Fabric")

axes[0][0].set_ylabel("Residents in \nOwner-Occupied Dwellings", fontsize=14)
axes[1][0].set_ylabel("Employment sector - \nAgriculture/Forestry/Fishery", fontsize=14)

axes[1][3].add_artist(ScaleBar(1, location="lower right", height_fraction=0.015))

# Colorbar for lc
cax1 = fig.add_axes((0.06, 0.2, 0.01, 0.6))
sm1 = mpl.cm.ScalarMappable(
    cmap=custom_coolwarm, norm=mpl.colors.Normalize(vmin=-1, vmax=1)
)
sm1.set_array([-1, 1])
cb1 = fig.colorbar(sm1, cax=cax1)
cb1.set_label("Local Coefficient")
cb1.ax.yaxis.set_label_position("left")

# Colorbar for fi
cax2 = fig.add_axes((0.905, 0.2, 0.01, 0.6))
sm2 = mpl.cm.ScalarMappable(
    cmap=custom_fi, norm=mpl.colors.Normalize(vmin=0, vmax=0.15)
)
sm2.set_array([0, 0.15])
cb2 = fig.colorbar(sm2, cax=cax2)
cb2.set_label("Feature Importance")
cb2.ax.yaxis.set_label_position("left")

fig.text(0.3, 0.92, "GWLR", ha="center", fontsize=16)
fig.text(0.7, 0.92, "GWRF", ha="center", fontsize=16)

fig.subplots_adjust(wspace=0, hspace=0)
fig.savefig("side_by_side.png", dpi=300, bbox_inches="tight")